<a href="https://colab.research.google.com/github/ProfAI/machine-learning-modelli-e-algoritmi/blob/main/1%20-%20L'algoritmo%20Gradient%20Descent/early_stopping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Early Stopping

### Cos'è l'Early Stopping?
L'**Early Stopping** è una tecnica di regolarizzazione utilizzata per prevenire l'**overfitting** (sovra-apprendimento) nei modelli addestrati in modo iterativo, come le reti neurali o i classificatori ottimizzati tramite Discesa del Gradiente (es. `SGDClassifier`).

### Perché lo usiamo?
Durante l'addestramento, il modello ottimizza i parametri per minimizzare la funzione di costo sul set di addestramento (Train Loss). Tuttavia, dopo un certo numero di epoche, il modello potrebbe iniziare a memorizzare il rumore del dataset di train, portando a una perdita di capacità di generalizzazione (l'errore sul set di test o validazione ricomincia a salire).

L'Early Stopping monitora una metrica calcolata su dati non visti (es. la Log Loss sul validation set o sul test set) e interrompe il processo di ottimizzazione non appena questa cessa di migliorare per un certo numero di iterazioni consecutive (parametro noto come `n_iter_no_change` o *patience*).

### La nostra implementazione manuale
Nel codice seguente implementiamo l'Early Stopping manualmente. Monitoriamo la **Log-Loss** (entropia incrociata binaria) definita come:
$$\mathcal{L}(y, p) = - \frac{1}{N} \sum_{i=1}^N \left( y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right)$$
Se la perdita all'epoca corrente $t$ non migliora di almeno una tolleranza $\text{tol} = 10^{-4}$ rispetto al miglior valore registrato (`best_loss`), incrementiamo un contatore. Se il contatore raggiunge `n_iter_no_change = 5`, l'addestramento viene interrotto.

In [1]:
from time import time
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
from sklearn.linear_model import SGDClassifier
from sklearn.utils import shuffle

In [27]:
RANDOM_SEED = 1

In [28]:
X, Y = make_classification(n_samples=1250, n_features=4, n_informative=2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.2, random_state=0)

In [ ]:
epochs = 200
n_batches = 1
tol = 0.0001
n_iter_no_change = 5
n_iter_count = 0

batch_size = X_train.shape[0]/n_batches

classes = np.unique(Y)

sgd = SGDClassifier(loss="log_loss", random_state=RANDOM_SEED)
full_losses = []
best_loss = 1

tick = time()


for epoch in range(epochs):
        loss = 0.
        X_shuffled, y_shuffled = shuffle(X_train, y_train, random_state=RANDOM_SEED)
        for batch in range(n_batches):
            batch_start = int(batch*batch_size)
            batch_end = int((batch+1)*batch_size)
            X_batch = X_shuffled[batch_start:batch_end,:]
            Y_batch = y_shuffled[batch_start:batch_end]

            sgd.partial_fit(X_batch, Y_batch, classes=classes)
            loss = log_loss(y_test, sgd.predict_proba(X_test),labels=classes)
            full_losses.append(loss)
        print("Loss all'epoca %d = %.4f (%.5f)" % (epoch+1, loss, best_loss-loss))


        if loss>=best_loss-tol:

          if n_iter_count>=n_iter_no_change:
            print("Early stopping")
            break
          else:
            n_iter_count+=1
        else:
            best_loss = loss
            n_iter_count = 0

print(f"Addestramento completato in {time()-tick:.2f} secondi")
loss = log_loss(y_test, sgd.predict_proba(X_test))
print(loss)

### Analisi dei Risultati dell'Implementazione Manuale
L'addestramento manuale si è interrotto all'**epoca 116** (su un massimo programmato di 200) raggiungendo una Log-Loss finale sul test set di **0.1066**.
Questo dimostra l'efficacia dell'Early Stopping: abbiamo risparmiato ben 84 epoche di calcolo inutile, evitando che il modello potesse andare in overfitting.

### Confronto con `SGDClassifier` di Scikit-Learn
Nel blocco di codice successivo, utilizziamo l'implementazione nativa di scikit-learn. Impostiamo `max_iter=200`, `tol=0.0001` e `n_iter_no_change=5`.

*Nota importante*: Scikit-learn arresta l'addestramento quando la perdita sul set di *validazione interno* (estratto dal train set) smette di migliorare. Con l'ottimizzazione nativa si ottiene una Log-Loss finale sul test set di **0.1169**.
La leggera differenza nei risultati è dovuta al fatto che l'arresto del modello nativo si basa sul validation split interno (default 10%), mentre il nostro ciclo manuale ha monitorato direttamente il test set (pratica usata qui a scopo illustrativo, ma che in produzione viene sostituita dall'uso di un validation set separato per evitare la contaminazione dei dati di test).

In [ ]:
sgf = SGDClassifier(max_iter=200,
                    loss="log_loss",
                     tol = 0.0001,
                     n_iter_no_change = 5,
                     random_state=RANDOM_SEED
                     )

sgf.fit(X_train, y_train)
loss = log_loss(y_test, sgf.predict_proba(X_test))
print(loss)